# Montagem — Dataset Final de Treino v1

Monta o `dataset_final_treino_v1.csv` a partir das camadas curated dos pipelines
`google_factcheck` e `noticias_reais` (Opção C).

**Composição:**
- **400 positivos (label=1):**
  - 26 claims verificados como VERDADEIRO pelo Google Fact Check (âncoras de formato)
  - 374 manchetes de notícias reais do RSS (filtradas e com cap de portal)
- **400 negativos (label=0):**
  - 260 FALSO + 120 ENGANOSO + 20 FORA_DE_CONTEXTO do Google Fact Check

**Regras:**
- Textos entre 50 e 400 caracteres em ambas as classes
- G1_POLITICA limitado a 120 registros nos positivos RSS
- Sem Fontes Oficiais nesta versão
- Seed fixa para reprodutibilidade
- Não altera nenhum arquivo raw ou curated existente

## Bibliotecas

In [1]:
import uuid
import pandas as pd
from pathlib import Path
from datetime import datetime

## Configuração

In [2]:
SEED = 42

# Limites de amostragem
N_TOTAL          = 400   # positivos e negativos
N_GFC_VERD       = 26    # âncoras GFC verdadeiro (todos os válidos)
N_RSS_ALVO       = N_TOTAL - N_GFC_VERD   # 374 — preenchimento RSS
CAP_G1           = 120   # máximo de registros G1_POLITICA nos positivos

N_NEG_FALSO      = 260
N_NEG_ENGANOSO   = 120
N_NEG_FORA_CTX   = 20

TAM_MIN = 50
TAM_MAX = 400

# Caminhos
PASTA_GFC_CURATED = Path("../dados/pipeline_falso_google_factcheck/curated")
PASTA_RSS_CURATED = Path("../dados/pipeline_noticias_reais/curated")
PASTA_SAIDA       = Path("../dados/dataset_unificado/final")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f"GFC curated : {PASTA_GFC_CURATED}")
print(f"RSS curated : {PASTA_RSS_CURATED}")
print(f"Saída       : {PASTA_SAIDA}")

GFC curated : ..\dados\pipeline_falso_google_factcheck\curated
RSS curated : ..\dados\pipeline_noticias_reais\curated
Saída       : ..\dados\dataset_unificado\final


## Carregamento dos dados curados

In [3]:
# Sempre usa o arquivo curated mais recente de cada pipeline
arq_gfc = sorted(PASTA_GFC_CURATED.glob("google_factcheck_curated_*.csv"))[-1]
arq_rss = sorted(PASTA_RSS_CURATED.glob("rss_noticias_curated_*.csv"))[-1]

df_gfc = pd.read_csv(arq_gfc, encoding="utf-8-sig", dtype=str)
df_rss = pd.read_csv(arq_rss, encoding="utf-8-sig", dtype=str)

print(f"GFC curated : {arq_gfc.name}  →  {len(df_gfc)} registros")
print(f"RSS curated : {arq_rss.name}  →  {len(df_rss)} registros")

GFC curated : google_factcheck_curated_2026-05-17_14-56-17.csv  →  3763 registros
RSS curated : rss_noticias_curated_2026-05-17_03-27-03.csv  →  617 registros


## Classe POSITIVA — Âncoras GFC VERDADEIRO

Inclui apenas os 26 claims verificados como verdadeiros pelo fact-checker,
excluindo os 4 registros com ressalvas identificadas na auditoria:
- 2 registros do OBSERVADOR (fact-checker português — fora do domínio brasileiro)
- "Comparação da liberdade de expressão nos EUA e Brasil..." (título de pauta, não claim)
- "Entenda como a modernização de cadastros da Reforma Tributária..." (manchete explicativa, não claim)

In [4]:
# Textos que não são claims verificáveis (identificados na auditoria manual)
_TEXTOS_EXCLUIR = {
    "Comparação da liberdade de expressão nos Estados Unidos e Brasil e embate entre Musk e Moraes",
    "Entenda como a modernização de cadastros da Reforma Tributária pode reajustar seu IPTU",
}

pos_gfc = (
    df_gfc
    [df_gfc["avaliacao_categoria"] == "VERDADEIRO"]
    # Exclui fact-checker português (fora do domínio)
    [df_gfc["fonte"] != "OBSERVADOR"]
    # Exclui títulos de pauta que não são claims
    [~df_gfc["texto_principal"].isin(_TEXTOS_EXCLUIR)]
    .copy()
)

assert len(pos_gfc) == N_GFC_VERD, (
    f"Esperados {N_GFC_VERD} GFC VERDADEIRO válidos, encontrados {len(pos_gfc)}"
)

print(f"GFC VERDADEIRO válidos: {len(pos_gfc)}")
print(pos_gfc[["avaliacao_original", "fonte", "texto_principal"]]
      .assign(chars=pos_gfc["texto_principal"].str.len())
      .to_string(index=False))

GFC VERDADEIRO válidos: 26
avaliacao_original            fonte                                                                                                                                                                                                                                                                                                                                                                                       texto_principal  chars
        Verdadeiro     UOL_NOTÍCIAS                                                                                                                                                                                                                                                                                                                                    Nunca se falou de contagem manual em discussão sobre voto impresso     66
        verdadeiro        AOS_FATOS                                                                        

C:\Users\offan\AppData\Local\Temp\ipykernel_20592\2338703349.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_gfc
C:\Users\offan\AppData\Local\Temp\ipykernel_20592\2338703349.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_gfc


## Classe POSITIVA — RSS Notícias Reais

Filtros aplicados:
- `flag_texto_longo == False`
- Texto entre 50 e 400 caracteres
- G1_POLITICA limitado a 120 registros (seed fixa)
- Amostragem final com seed fixa para atingir 374 registros

In [5]:
df_rss["_tam"] = df_rss["texto_principal"].str.len()

rss_filtrado = df_rss[
    (df_rss["flag_texto_longo"].str.strip().str.lower() == "false") &
    (df_rss["_tam"] >= TAM_MIN) &
    (df_rss["_tam"] <= TAM_MAX)
].copy()

print(f"RSS após filtro de tamanho e flag: {len(rss_filtrado)} registros")
print(f"  G1_POLITICA disponível: {(rss_filtrado['portal'] == 'G1_POLITICA').sum()}")

# Separa G1 e não-G1
rss_g1     = rss_filtrado[rss_filtrado["portal"] == "G1_POLITICA"]
rss_nao_g1 = rss_filtrado[rss_filtrado["portal"] != "G1_POLITICA"]

# Cap G1
rss_g1_cap = rss_g1.sample(n=min(CAP_G1, len(rss_g1)), random_state=SEED)
print(f"  G1_POLITICA após cap ({CAP_G1}): {len(rss_g1_cap)}")
print(f"  Não-G1 disponível: {len(rss_nao_g1)}")

# Pool RSS pós-cap
rss_pool = pd.concat([rss_g1_cap, rss_nao_g1], ignore_index=True)
print(f"  Pool RSS pós-cap: {len(rss_pool)} disponíveis para {N_RSS_ALVO} necessários")

assert len(rss_pool) >= N_RSS_ALVO, (
    f"Pool RSS insuficiente: {len(rss_pool)} < {N_RSS_ALVO}"
)

pos_rss = rss_pool.sample(n=N_RSS_ALVO, random_state=SEED).copy()
print(f"\nRSS amostrado (seed={SEED}): {len(pos_rss)}")
print("Distribuição por portal:")
print(pos_rss["portal"].value_counts().to_string())

RSS após filtro de tamanho e flag: 580 registros
  G1_POLITICA disponível: 276
  G1_POLITICA após cap (120): 120
  Não-G1 disponível: 304
  Pool RSS pós-cap: 424 disponíveis para 374 necessários

RSS amostrado (seed=42): 374
Distribuição por portal:
portal
G1_POLITICA            109
FOLHA_PODER             60
PODER360                33
BBC_BRASIL              30
AGENCIA_BRASIL          27
CORREIO_BRAZILIENSE     26
UOL_NOTICIAS            20
CARTACAPITAL            19
VEJA_POLITICA           17
METROPOLES              17
CONGRESSO_EM_FOCO       16


## Classe POSITIVA — Combinação e mapeamento de schema

In [6]:
def mapear_pos_gfc(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "id_registro"   : df["id_registro"].values,
        "texto_principal": df["texto_principal"].values,
        "label"         : 1,
        "label_detalhe" : "GFC_VERDADEIRO",
        "pipeline_origem": "google_factcheck",
        "portal_origem"  : df["fonte"].values,
        "origem_texto"   : "afirmacao_checada",
        "tamanho_chars"  : df["texto_principal"].str.len().values,
        "data_publicacao": df["data_publicacao"].values,
        "url_origem"     : df["url_origem"].values,
    })


def mapear_pos_rss(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "id_registro"   : df["id_registro"].values,
        "texto_principal": df["texto_principal"].values,
        "label"         : 1,
        "label_detalhe" : "NOTICIA_REAL",
        "pipeline_origem": "noticias_reais",
        "portal_origem"  : df["portal"].values,
        "origem_texto"   : df["origem_texto"].values,
        "tamanho_chars"  : df["texto_principal"].str.len().values,
        "data_publicacao": df["data_publicacao"].values,
        "url_origem"     : df["url_origem"].values,
    })


positivos = pd.concat(
    [mapear_pos_gfc(pos_gfc), mapear_pos_rss(pos_rss)],
    ignore_index=True
)

assert len(positivos) == N_TOTAL, f"Esperados {N_TOTAL} positivos, obtidos {len(positivos)}"
print(f"Total de positivos: {len(positivos)}")
print(f"  GFC_VERDADEIRO : {(positivos['label_detalhe'] == 'GFC_VERDADEIRO').sum()}")
print(f"  NOTICIA_REAL   : {(positivos['label_detalhe'] == 'NOTICIA_REAL').sum()}")

Total de positivos: 400
  GFC_VERDADEIRO : 26
  NOTICIA_REAL   : 374


## Classe NEGATIVA — GFC FALSO / ENGANOSO / FORA_DE_CONTEXTO

Filtros:
- `avaliacao_categoria` em {FALSO, ENGANOSO, FORA_DE_CONTEXTO}
- Texto entre 50 e 400 caracteres
- Amostragem estratificada com seed fixa: 260 + 120 + 20 = 400

In [7]:
df_gfc["_tam"] = df_gfc["texto_principal"].str.len()

_cats_neg = {"FALSO", "ENGANOSO", "FORA_DE_CONTEXTO"}

gfc_neg_pool = df_gfc[
    (df_gfc["avaliacao_categoria"].isin(_cats_neg)) &
    (df_gfc["_tam"] >= TAM_MIN) &
    (df_gfc["_tam"] <= TAM_MAX)
].copy()

print("Pool negativo disponível (50-400 chars):")
print(gfc_neg_pool["avaliacao_categoria"].value_counts().to_string())

_cotas = {
    "FALSO"           : N_NEG_FALSO,
    "ENGANOSO"        : N_NEG_ENGANOSO,
    "FORA_DE_CONTEXTO": N_NEG_FORA_CTX,
}

partes_neg = []
for cat, n in _cotas.items():
    pool_cat = gfc_neg_pool[gfc_neg_pool["avaliacao_categoria"] == cat]
    assert len(pool_cat) >= n, f"Pool insuficiente para {cat}: {len(pool_cat)} < {n}"
    partes_neg.append(pool_cat.sample(n=n, random_state=SEED))
    print(f"  {cat}: {n} amostrados de {len(pool_cat)} disponíveis")

neg_gfc = pd.concat(partes_neg, ignore_index=True)
print(f"\nTotal negativos: {len(neg_gfc)}")

Pool negativo disponível (50-400 chars):
avaliacao_categoria
FALSO               2141
ENGANOSO             642
FORA_DE_CONTEXTO      67
  FALSO: 260 amostrados de 2141 disponíveis
  ENGANOSO: 120 amostrados de 642 disponíveis
  FORA_DE_CONTEXTO: 20 amostrados de 67 disponíveis

Total negativos: 400


## Classe NEGATIVA — Mapeamento de schema

In [8]:
def mapear_neg_gfc(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "id_registro"   : df["id_registro"].values,
        "texto_principal": df["texto_principal"].values,
        "label"         : 0,
        "label_detalhe" : df["avaliacao_categoria"].values,
        "pipeline_origem": "google_factcheck",
        "portal_origem"  : df["fonte"].values,
        "origem_texto"   : "afirmacao_checada",
        "tamanho_chars"  : df["texto_principal"].str.len().values,
        "data_publicacao": df["data_publicacao"].values,
        "url_origem"     : df["url_origem"].values,
    })


negativos = mapear_neg_gfc(neg_gfc)

assert len(negativos) == N_TOTAL, f"Esperados {N_TOTAL} negativos, obtidos {len(negativos)}"
print(f"Total de negativos: {len(negativos)}")
print(negativos["label_detalhe"].value_counts().to_string())

Total de negativos: 400
label_detalhe
FALSO               260
ENGANOSO            120
FORA_DE_CONTEXTO     20


## Montagem e exportação do dataset final

In [9]:
COLUNAS_FINAIS = [
    "id_registro",
    "texto_principal",
    "label",
    "label_detalhe",
    "pipeline_origem",
    "portal_origem",
    "origem_texto",
    "tamanho_chars",
    "data_publicacao",
    "url_origem",
]

df_final = (
    pd.concat([positivos, negativos], ignore_index=True)
    [COLUNAS_FINAIS]
    # Embaralha mantendo reprodutibilidade
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

caminho_saida = PASTA_SAIDA / "dataset_final_treino_v1.csv"
df_final.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Dataset salvo em: {caminho_saida}")
print(f"Total de registros: {len(df_final)}")

Dataset salvo em: ..\dados\dataset_unificado\final\dataset_final_treino_v1.csv
Total de registros: 800


## Resumo e verificação de qualidade

In [10]:
print("=" * 60)
print("RESUMO — dataset_final_treino_v1")
print("=" * 60)

print(f"\n1. TOTAL DE REGISTROS: {len(df_final)}")

print("\n2. CONTAGEM POR LABEL:")
print(df_final["label"].value_counts().rename({0: "0 (negativo)", 1: "1 (positivo)"}).to_string())

print("\n3. CONTAGEM POR LABEL_DETALHE:")
print(df_final["label_detalhe"].value_counts().to_string())

print("\n4. CONTAGEM POR PORTAL_ORIGEM (top 20):")
print(df_final["portal_origem"].value_counts().head(20).to_string())

print("\n5. ESTATÍSTICAS DE TAMANHO POR CLASSE:")
stats = df_final.groupby("label")["tamanho_chars"].describe()[["min","50%","mean","max"]]
stats.index = stats.index.map({0: "0-negativo", 1: "1-positivo"})
print(stats.round(1).to_string())

print("\n6a. 10 EXEMPLOS POSITIVOS (label=1):")
for i, (_, r) in enumerate(
    df_final[df_final["label"] == 1].sample(10, random_state=SEED).iterrows()
):
    print(f"  [{i+1}] [{r['label_detalhe']}] [{r['portal_origem']}] "
          f"{r['tamanho_chars']} chars")
    print(f"       {repr(r['texto_principal'][:120])}")

print("\n6b. 10 EXEMPLOS NEGATIVOS (label=0):")
for i, (_, r) in enumerate(
    df_final[df_final["label"] == 0].sample(10, random_state=SEED).iterrows()
):
    print(f"  [{i+1}] [{r['label_detalhe']}] [{r['portal_origem']}] "
          f"{r['tamanho_chars']} chars")
    print(f"       {repr(r['texto_principal'][:120])}")

print("\n7. ORIGEM DOS POSITIVOS:")
print(df_final[df_final["label"] == 1]["label_detalhe"].value_counts().to_string())

print("\n   Pipeline de origem dos positivos:")
print(df_final[df_final["label"] == 1]["pipeline_origem"].value_counts().to_string())

print("\n   origem_texto dos positivos:")
print(df_final[df_final["label"] == 1]["origem_texto"].value_counts().to_string())

RESUMO — dataset_final_treino_v1

1. TOTAL DE REGISTROS: 800

2. CONTAGEM POR LABEL:
label
0 (negativo)    400
1 (positivo)    400

3. CONTAGEM POR LABEL_DETALHE:
label_detalhe
NOTICIA_REAL        374
FALSO               260
ENGANOSO            120
GFC_VERDADEIRO       26
FORA_DE_CONTEXTO     20

4. CONTAGEM POR PORTAL_ORIGEM (top 20):
portal_origem
ESTADÃO                110
G1_POLITICA            109
AOS_FATOS              105
UOL_NOTÍCIAS            77
AFP_CHECAMOS            64
FOLHA_PODER             60
PODER360                33
BBC_BRASIL              30
AGENCIA_BRASIL          27
CORREIO_BRAZILIENSE     26
BOATOS.ORG              24
DESCONHECIDA            22
UOL_NOTICIAS            20
CARTACAPITAL            19
METROPOLES              17
VEJA_POLITICA           17
CONGRESSO_EM_FOCO       16
PROJETO_COMPROVA         9
BOL_-_UOL                7
FOLHA_-_UOL              5

5. ESTATÍSTICAS DE TAMANHO POR CLASSE:
             min    50%   mean    max
label                         